# Libs

In [ ]:
import os
import json
import re
import io
import soundfile as sf
import random
from collections import OrderedDict
import math
from collections import Counter

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model

from tensorflow.keras.layers import *
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
from tensorflow.keras.utils import plot_model, Sequence

In [ ]:
import glob
import pickle
import tqdm
import kagglehub

In [ ]:
import librosa
from IPython.display import Audio as AudioPlayer, display

# Constants

In [ ]:
BASE_PATH = r"tts_kmeans_seq2seq"
os.makedirs(BASE_PATH, exist_ok=True)

TXT_TOKENS_CACHE_PATH = os.path.join(BASE_PATH, 'txt_tokens.json')
MEL_TOKENS_CACHE_PATH = os.path.join(BASE_PATH, 'mel_tokens.json')

TRAIN_CACHE_DIR = os.path.join(BASE_PATH, 'train_cache')
os.makedirs(TRAIN_CACHE_DIR, exist_ok=True)

In [ ]:
BATCH_SIZE = 6

In [ ]:
TEXT_MAX_LENGTH = 300
LOG_MEL_MAX_LENGTH = 1200

In [ ]:
TOKEN_PAD = 'PAD'

TOKEN_START = 'START'
TOKEN_END = 'END'

# Functions

In [ ]:
def pkl_save_obj(obj, file_name):
    with open(file_name, 'wb') as f:
        pickle.dump(obj, f)

def pkl_load_obj(file_name):
    with open(file_name, 'rb') as f:
        return pickle.load(f)

In [ ]:
def json_save_obj(obj, file_name):
    with open(file_name, 'w', encoding='UTF+8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def json_load_obj(file_name):
    with open(file_name, 'r', encoding='UTF+8') as f:
        return json.load(f)

# Dataset

In [ ]:
path = kagglehub.dataset_download("youssef019/da7ee7-audio-text-with-clustered-log-mel-pins")

print("Path to dataset files:", path)

In [ ]:
os.listdir(path)

In [ ]:
data_df = pd.read_csv(os.path.join(path, "data_table.csv"))

In [ ]:
config_dict = json_load_obj(os.path.join(path, "config.json"))

AUDIO_SR = config_dict['sample_rate']
AUDIO_N_FFT = config_dict['n_fft']
AUDIO_N_MELS = config_dict['n_mels']
AUDIO_HOP_LENGTH = config_dict['hop_length']
AUDIO_WINDOW_TYPE = config_dict['window_type']

In [ ]:
print("sample rare:", AUDIO_SR)
print("n_fft:", AUDIO_N_FFT)
print("n_mels:", AUDIO_N_MELS)
print("hop_length:", AUDIO_HOP_LENGTH)
print("window_type:", AUDIO_WINDOW_TYPE)

# TXT Preprocessing

In [ ]:
data_df.head()

## Tokens

In [ ]:
tokens_dict = {
    TOKEN_PAD: 0,

    "ا": 1,
    "أ": 1,
    "إ": 1,
    "آ": 1,
    "ء": 2,
    "ب": 3,
    "ت": 4,
    "ث": 5,
    "ج": 6,
    "ح": 7,
    "خ": 8,
    "د": 9,
    "ذ": 10,
    "ر": 11,
    "ز": 12,
    "س": 13,
    "ش": 14,
    "ص": 15,
    "ض": 16,
    "ط": 17,
    "ظ": 18,
    "ع": 19,
    "غ": 20,
    "ف": 21,
    "ڤ": 21,
    "ق": 24,
    "ك": 25,
    "ک": 25,
    "ڪ": 25,
    "ل": 26,
    "م": 27,
    "ن": 28,
    "ه": 29,
    "ة": 29,
    "و": 30,
    "ي": 31,
    "ى": 31,
    "ئ": 32,
    "ؤ": 33,
    " ": 34,
    "\n": 34,
    "\t": 34,
}


In [ ]:
reverse_tokens_dict = {v: k for k, v in tokens_dict.items()}

In [ ]:
json_save_obj(tokens_dict, TXT_TOKENS_CACHE_PATH)

In [ ]:
TXT_N_TOKENS = max(tokens_dict.values()) + 1
print("Text N Tokens:", TXT_N_TOKENS)

## Preprocessing

In [ ]:
def process_txt(txt:str):
    txt = str(txt)
    txt = txt.lower()

    txt = re.sub(r'\s+', ' ', txt)
    txt = txt.strip()

    clean_txt = ''
    for char in txt:
        if char in tokens_dict:
            clean_txt += char

    return clean_txt

In [ ]:
data_df['cleaned_txt'] = data_df['text'].apply(process_txt)

In [ ]:
data_df['txt_len'] = data_df['cleaned_txt'].apply(len)

In [ ]:
data_df[['cleaned_txt', 'text']]

In [ ]:
sns.histplot(data_df, x='txt_len')

## Lenght Filtering

In [ ]:
print("Length before:", len(data_df))

In [ ]:
data_df = data_df[ data_df['txt_len'] <= TEXT_MAX_LENGTH ]
data_df.reset_index(drop=True, inplace=True)

In [ ]:
print("Length after:", len(data_df))

## Tokenization

In [ ]:
def tokenize(txt:str, max_length=TEXT_MAX_LENGTH):
    output = []

    pad_id = tokens_dict[TOKEN_PAD]

    # tokens_dict must return a value, all texts r cleaned
    output = list(map(lambda x: tokens_dict.get(x), txt))

    output = output[:max_length]

    output = np.array(output, dtype=int)
    return output

In [ ]:
data_df['txt_tokens_idx'] = data_df['cleaned_txt'].apply(tokenize)

In [ ]:
data_df['txt_tokens_idx'][0]

# LOG MEL Preprocessing

## Tokens

In [ ]:
audio_tokens_dict = {
    TOKEN_PAD:0,
    TOKEN_START:1,
    TOKEN_END:2
}

In [ ]:
temp_set = set()

for i in range(len(data_df)):
    temp_tokens = str(data_df.loc[i, "centroids"])
    temp_set.update(temp_tokens.split(' '))

temp_set = list(temp_set)
temp_set.sort()

for token in temp_set:
    audio_tokens_dict[token] = len(audio_tokens_dict)

In [ ]:
AUDIO_N_TOKENS = len(audio_tokens_dict)

In [ ]:
print("N Log Mel Tokens:", AUDIO_N_TOKENS)

In [ ]:
list(audio_tokens_dict.keys())[:12]

In [ ]:
json_save_obj(audio_tokens_dict, MEL_TOKENS_CACHE_PATH)

In [ ]:
reverse_audio_tokens_dict = {v: k for k, v in audio_tokens_dict.items()}

## Length Filtering

In [ ]:
data_df['log_mel_len'] = data_df['centroids'].apply(
    lambda x: len(x.split(' '))
)

In [ ]:
sns.histplot(data_df, x='log_mel_len')

In [ ]:
print("Length before:", len(data_df))

In [ ]:
data_df = data_df[ data_df['log_mel_len'] <= LOG_MEL_MAX_LENGTH ]
data_df.reset_index(drop=True, inplace=True)

In [ ]:
print("Length after:", len(data_df))

## Tokenization

In [ ]:
def tokenize_audio(txt:str, max_length=LOG_MEL_MAX_LENGTH):
    output = []

    pad_id = audio_tokens_dict[TOKEN_PAD]
    start_id = audio_tokens_dict[TOKEN_START]
    end_id = audio_tokens_dict[TOKEN_END]

    # Remove 1, to make some room for start, end tokens
    max_length = max_length - 1

    txt_splitted = txt.split(' ')

    # tokens_dict must return a value, all texts r cleaned
    output = list(map(lambda x: audio_tokens_dict.get(x), txt_splitted))

    output = output[:max_length]

    output = np.array(output, dtype=int)

    return output

In [ ]:
data_df['log_mel_tokens_idx'] = data_df['centroids'].apply(tokenize_audio)

In [ ]:
list(data_df.loc[0,'log_mel_tokens_idx'])[:35]

# Audio Functions

## Centroids

In [ ]:
centroid_path = os.path.join(path, 'clusters_points.json')
centroid_path

In [ ]:
centroids_json = json_load_obj(centroid_path)
print("Len Centroids:", len(centroids_json))

## Tokens Idx 2 Tokens

In [ ]:
def audio_tokens_idx_2_tokens(audio_tokens_idx):
    audio_tokens = []

    pad_id = audio_tokens_dict[TOKEN_PAD]
    start_id = audio_tokens_dict[TOKEN_START]
    end_id = audio_tokens_dict[TOKEN_END]

    for token_idx in audio_tokens_idx:
        if token_idx == start_id:
            continue
        elif token_idx == end_id:
            break
        elif token_idx == pad_id:
            continue
        else:
            audio_tokens.append(
                reverse_audio_tokens_dict[int(token_idx)]
            )
    return audio_tokens

In [ ]:
audio_tokens_idx_2_tokens([3,4,5,1])

## Tokens 2 Log mel

In [ ]:
def audio_tokens_2_log_mels_values(audio_tokens):
    log_mel_pins = []

    for token_idx in audio_tokens:
        if token_idx == TOKEN_START:
            continue
        elif token_idx == TOKEN_END:
            break
        elif token_idx == TOKEN_PAD:
            continue
        else:
            log_mel_pins.append(
                centroids_json[token_idx]
            )
    log_mel_pins = np.array(log_mel_pins)
    log_mel_pins = log_mel_pins.T # Freq, Time
    return log_mel_pins

In [ ]:
temp = audio_tokens_2_log_mels_values(['pin_0000', 'pin_0001', 'pin_0002'])
temp.shape

## Log mel 2 Audio

In [ ]:
def log_mel_2_audio(log_mel):
    # Convert back to power (dB -> amplitude)
    mel_spec = librosa.db_to_power(log_mel)

    # Invert Mel spectrogram to audio
    y = librosa.feature.inverse.mel_to_audio(
        M=mel_spec,
        sr=AUDIO_SR,
        n_fft=AUDIO_N_FFT,
        hop_length=AUDIO_HOP_LENGTH,
        win_length=AUDIO_N_FFT,
        window=AUDIO_WINDOW_TYPE,
    )

    return y.astype(np.float32)

## Test Pipeline

In [ ]:
def test_pipe(idx:int):
    cleaned_txt = data_df.loc[idx, 'cleaned_txt']
    print("Text:", cleaned_txt)

    audio_tokens_idx = data_df.loc[idx, 'log_mel_tokens_idx']

    audio_tokens = audio_tokens_idx_2_tokens(audio_tokens_idx)
    print("Audio Tokens(first 35):", audio_tokens[:35])

    log_mel = audio_tokens_2_log_mels_values(audio_tokens)
    print("Quantized Log Mel Shape:", log_mel.shape)

    librosa.display.specshow(log_mel, sr=AUDIO_SR, hop_length=AUDIO_HOP_LENGTH,
                         x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.2f dB')
    plt.title('Quantized Log-Mel Spectrogram')
    plt.tight_layout()
    plt.show()

    audio = log_mel_2_audio(log_mel)
    print("Audio Shape:", audio.shape)

    display(AudioPlayer(audio, rate=AUDIO_SR))

In [ ]:
test_pipe(0)

In [ ]:
test_pipe(1)

# Improvements

## Class Weights

In [ ]:
# 1. Collect all token IDs used in targets (as integers)
all_tokens = []
for tokens_array in data_df['log_mel_tokens_idx']:   # already numpy arrays of ints
    all_tokens.extend(tokens_array)

cnt = Counter(all_tokens)
print("Most common tokens:", cnt.most_common(10))
print("Total unique tokens found in data:", len(cnt))
print("Vocabulary size:", len(audio_tokens_dict))

In [ ]:
# 2. Create class weight array (index = token_id)
class_weight_arr = np.ones(len(audio_tokens_dict), dtype=np.float32)

total_nonpad_tokens = sum(cnt.values())   # excludes PAD if PAD not in targets

for token_id, count in cnt.items():
    # inverse frequency
    class_weight_arr[token_id] = total_nonpad_tokens / (count + 1.0)

In [ ]:
# give special tokens custom weights
pad_id   = audio_tokens_dict['PAD']
start_id = audio_tokens_dict['START']
end_id   = audio_tokens_dict['END']

class_weight_arr[pad_id]   = 0.05     # very low — we mostly ignore PAD anyway
class_weight_arr[start_id] = 1.0
class_weight_arr[end_id]   = 2.0      # slightly encourage producing END

In [ ]:
# Normalize so mean weight ≈ 1.0 (helps keep learning rate scale similar)
mean_weight = np.mean(class_weight_arr[class_weight_arr > 0.1])  # exclude pad if very low
class_weight_arr = class_weight_arr / mean_weight

In [ ]:
# Quick debug prints
most_freq_id = cnt.most_common(1)[0][0]
least_freq_id = cnt.most_common()[-1][0]

print("\nAfter weighting:")
print(f"Most frequent token id: {most_freq_id}  → weight = {class_weight_arr[most_freq_id]:.4f}")
print(f"Least frequent token id: {least_freq_id} → weight = {class_weight_arr[least_freq_id]:.4f}")
print(f"PAD weight   : {class_weight_arr[pad_id]:.6f}")
print(f"START weight : {class_weight_arr[start_id]:.6f}")
print(f"END weight   : {class_weight_arr[end_id]:.6f}")

In [ ]:
CLASS_WEIGHTS = tf.constant(class_weight_arr, dtype=tf.float32)

#  Data Generator

In [ ]:
class MyDataGenerator(Sequence):
    def __init__(self, gen_loader_df, augment=True, shuffle=True):
        self.gen_loader_df = gen_loader_df
        self.gen_loader_df.reset_index(drop=True, inplace=True)

        self.batch_size = BATCH_SIZE
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(self.gen_loader_df))
        self.on_epoch_end()

        self.x_pad_id = tokens_dict[TOKEN_PAD]
        self.y_pad_id = audio_tokens_dict[TOKEN_PAD]
        self.y_start_id = audio_tokens_dict[TOKEN_START]
        self.y_end_id = audio_tokens_dict[TOKEN_END]

    def __len__(self):
        # Number of batches per epoch
        original_lenght = int(np.ceil(len(self.gen_loader_df) / self.batch_size))
        return original_lenght

    def __getitem__(self, idx):
        # Compute batch indices
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        # Load batch data
        batch_x_tokens = [self._load_x_tokens(i) for i in batch_indices]
        batch_y_tokens_as_input = [self._load_y_tokens_as_input(i) for i in batch_indices]
        batch_y_tokens_as_output = [self._load_y_tokens_as_output(i) for i in batch_indices]

        # Convert to numpy arrays
        batch_x_tokens = np.array(batch_x_tokens, dtype=int)
        batch_y_tokens_as_input = np.array(batch_y_tokens_as_input, dtype=int)
        batch_y_tokens_as_output = np.array(batch_y_tokens_as_output, dtype=int)

        return (batch_x_tokens, batch_y_tokens_as_input), batch_y_tokens_as_output

    def on_epoch_end(self):
        # Shuffle indices after each epoch
        if self.shuffle:
            np.random.shuffle(self.indices)

    def _load_x_tokens(self, idx):
         tokens_x = self.gen_loader_df.loc[idx, 'txt_tokens_idx']
         # Add padding
         tokens_x = np.pad(tokens_x,
         (0, TEXT_MAX_LENGTH - len(tokens_x)),
         'constant',
         constant_values=self.x_pad_id)

         return tokens_x

    def _load_y_tokens_as_input(self, idx):
        tokens_y = self.gen_loader_df.loc[idx, 'log_mel_tokens_idx']

        # Add Start token (one time)
        tokens_y = np.pad(
            tokens_y,
            (1, 0),
            'constant',
            constant_values=self.y_start_id
        )

        # Fill with padding
        tokens_y = np.pad(
            tokens_y,
            (0, LOG_MEL_MAX_LENGTH - len(tokens_y)),
            'constant',
            constant_values=self.y_pad_id
        )

        return tokens_y

    def _load_y_tokens_as_output(self, idx):
        tokens_y = self.gen_loader_df.loc[idx, 'log_mel_tokens_idx']

        # Add End token (one time)
        tokens_y = np.pad(
            tokens_y,
            (0, 1),
            'constant',
            constant_values=self.y_end_id
        )

        # Fill with padding
        tokens_y = np.pad(
            tokens_y,
            (0, LOG_MEL_MAX_LENGTH - len(tokens_y)),
            'constant',
            constant_values=self.y_pad_id
        )

        return tokens_y


In [ ]:
train_data_gen = MyDataGenerator(
    data_df,
    augment=True,
    shuffle=True
)

samples_data_gen = MyDataGenerator(
    data_df.iloc[:10,:], # few rows for testing
    augment=True,
    shuffle=True
)

# val_data_gen = MyDataGenerator(
#     val_loader_df,
#     augment=False,
#     shuffle=True
# )

# test_data_gen = MyDataGenerator(
#     test_loader_df,
#     augment=False,
#     shuffle=True
# )

In [ ]:
for i in [0]:
    batch_i = train_data_gen[i]
    (batch_x_tokens, batch_y_tokens_as_input), batch_y_tokens_as_output = batch_i

    print("Input Shape:", batch_x_tokens.shape[1:])
    print("Output shape I/P:", batch_y_tokens_as_input.shape[1:])
    print("Output shape O/P:", batch_y_tokens_as_output.shape[1:])
    print()
    print("I/P X:", list(batch_x_tokens[i]))
    print()
    print("I/P Y:", list(batch_y_tokens_as_input[i]))
    print()
    print("O/P Y:", list(batch_y_tokens_as_output[i]))

    print("\n"*4)

# Model Encapsulation

In [ ]:
class HistoryCheckpoint(Callback):
    def __init__(self, model_name):
        super().__init__()
        self.model_name = model_name
        self.filepath = os.path.join(TRAIN_CACHE_DIR, f'{model_name}.history.json')
        self.history = {}

        if os.path.exists(self.filepath):
            with open(self.filepath, 'r') as f:
                self.history = json.load(f)
                print(f'Model {self.model_name} history loaded from cache')
        else:
            self.history = {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for key, value in logs.items():
            self.history.setdefault(key, []).append(float(value))

        with open(self.filepath, 'w') as f:
            json.dump(self.history, f)

    def get_history(self):
        return self.history

In [ ]:
class CustomModel:
    def __init__(self, name, entire_model, encoder_model, decoder_model):
        self.name = re.sub('[^A-Za-z0-9]', '_', str(name)).lower()

        self.entire_model = entire_model
        self.encoder_model = encoder_model
        self.decoder_model = decoder_model

        self.trained = False
        self._history = HistoryCheckpoint(self.name).get_history()

        self.load()

    def fit(self, data_gen, validation_data, epochs, force_train= False):
        file_path = os.path.join(TRAIN_CACHE_DIR, self.name + '.weights.h5')

        es = EarlyStopping(
            monitor='loss',
            patience=10,
            verbose=1,
            restore_best_weights=True,
            mode='min'
        )

        mcp = ModelCheckpoint(
            filepath=file_path,
            monitor='loss',
            save_best_only=True,
            save_weights_only=True,
            mode='min',
            verbose=1
        )

        rop = ReduceLROnPlateau(monitor='loss', factor=0.1, patience=2)

        hcp = HistoryCheckpoint(self.name)

        if self.trained == False or force_train:
            initial_epoch = len(hcp.get_history().get('loss', []))
            if initial_epoch >= epochs:
                print(f"Model {self.name} has already trained for {epochs} epochs")
                self._history = hcp.get_history()
                return

            self.entire_model.fit(
                data_gen,
                validation_data=validation_data,
                callbacks=[es, mcp, rop, hcp],
                epochs=epochs,
                initial_epoch=initial_epoch
            )

            self.trained = True
            self._history = hcp.get_history()
        else:
             self._history = hcp.get_history()


    @property
    def history(self):
        return self._history

    def load(self):
        file_path = os.path.join(TRAIN_CACHE_DIR, self.name + '.weights.h5')

        if os.path.exists(file_path):
            self.entire_model.load_weights(file_path)
            self.trained = True
            print(f'Model {self.name} weights loaded from cache')
        else:
            self.trained = False
            print(f'Model {self.name} weights not found in cache, training required')

# Model Building

## Loss

In [ ]:
def masked_categorical_crossentropy(y_true, y_pred):
    """
    Weighted version - rare tokens get higher loss contribution
    Ignores PAD tokens (token == 0)
    """
    y_true = tf.cast(y_true, tf.int32)           # (batch, seq_len)
    mask   = tf.cast(tf.not_equal(y_true, 0), tf.float32)

    # Gather per-token weights according to true label
    per_token_weights = tf.gather(CLASS_WEIGHTS, y_true)   # (batch, seq_len)

    y_true_one_hot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])

    # Standard cross-entropy per position
    ce = tf.keras.losses.categorical_crossentropy(
        y_true_one_hot,
        y_pred,
        from_logits=False
    )  # shape: (batch, seq_len)

    # Apply class weights + masking
    weighted_loss = ce * per_token_weights * mask

    # Normalize by number of **valid** (non-pad) tokens
    num_valid = tf.reduce_sum(mask) + 1e-8
    return tf.reduce_sum(weighted_loss) / num_valid

In [ ]:
def masked_categorical_accuracy(y_true, y_pred):
    """
    y_true: (batch, seq_len) integer token indices
    y_pred: (batch, seq_len, num_classes) softmax probabilities
    Ignores PAD tokens (value = 0)
    """
    # Make sure y_true is int
    y_true = tf.cast(y_true, tf.int32)

    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)

    y_pred_class = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

    matches = tf.cast(tf.equal(y_true, y_pred_class), tf.float32)

    matches *= mask

    return tf.reduce_sum(matches) / tf.reduce_sum(mask)

## Layers

In [ ]:
def parallel_cnn_1d_layers(
    input_tensor,
    n_filters,
    kernel_sizes,
    concate_filters,
    skip_conn,
    name,
    norm: bool,
    activation
):
    outputs = []

    for i in range(len(kernel_sizes)):
        conv_out = Conv1D(
            filters=n_filters[i],
            kernel_size=kernel_sizes[i],
            activation=activation,
            padding='same',
            name=f"{name}_cnn{i}"
        )(input_tensor)

        outputs.append(conv_out)

    if skip_conn:
        outputs.append(input_tensor)

    # Concatenate along channel axis (last axis)
    signal = Concatenate(axis=-1, name=f"{name}_concat")(outputs)

    # 1x1 Conv1D = kernel_size=1
    single_out = Conv1D(
        filters=concate_filters,
        kernel_size=1,
        activation='linear' if norm else 'relu',
        padding='same',
        name=f"{name}_merge"
    )(signal)

    if norm:
        single_out = LayerNormalization(name=f"{name}_norm")(single_out)
        single_out = Activation('relu', name=f"{name}_act")(single_out)

    return single_out

In [ ]:
def parallel_cnn_2d_layers(
    input_tensor,
    n_filters,
    kernel_sizes,
    concate_filters,
    skip_conn,
    name,
    norm:bool,
    activation
):
    outputs = []

    for i in range(len(kernel_sizes)):
        conv_out = Conv2D(
            n_filters[i],
            kernel_sizes[i],
            activation=activation,
            padding='same',
            name=f"{name}_cnn{i}")(input_tensor)
        outputs.append(conv_out)


    if skip_conn:
        outputs.append(input_tensor)

    # Concatenate along channel axis
    signal = Concatenate(axis=-1, name=f"{name}_concat")(outputs)

    # Reduce/merge filters with 1x1 convolution
    single_out = Conv2D(
        concate_filters,
         (1, 1),
        activation='linear' if norm else "relu",
        padding='same',
        name=f"{name}_merge")(signal)

    if norm:
        single_out = LayerNormalization(name=f"{name}_norm")(single_out)
        single_out = Activation('relu', name=f"{name}_act")(single_out)

    return single_out

## Models

In [ ]:
models_list = []

## Model 1

In [ ]:
M1_LSTM_CELLS = 8 * 84
M1_EMBEDDING_DIM = 72

In [ ]:
def build_m1_encoder():
    input = Input(shape=(TEXT_MAX_LENGTH,), name="enc_input")

    emb_1 = Embedding(TXT_N_TOKENS, M1_EMBEDDING_DIM)(input)

    emb_2 = parallel_cnn_1d_layers(
        input_tensor=emb_1,
        n_filters=[64, 32, 16, 8, 4],
        kernel_sizes=[3, 5, 7, 9, 15],
        concate_filters=M1_EMBEDDING_DIM,
        skip_conn=False,
        name="enc_cnn_block_1",
        norm=False,
        activation='relu'
    )

    out_1, h_1, c_1  = LSTM(
      M1_LSTM_CELLS,
      return_sequences=True,
      return_state=True,
      name='enc_lstm_1'
    )(emb_2)

    out_2, h_2, c_2  = LSTM(
      M1_LSTM_CELLS,
      return_sequences=True,
      return_state=True,
      name='enc_lstm_2'
    )(out_1)


    model = Model(
        input,
        [out_2, h_1, c_1, h_2, c_2],
        name='encoder_model')

    model.summary()

    return model

In [ ]:
def build_m1_decoder():
    M1_DEC_LSTM_CELLS = 8 * 84

    M1_DEC_EMBEDDING_DIM = 512

    NUM_HEADS = 8  # Multi-head attention
    KEY_DIM = M1_DEC_LSTM_CELLS // NUM_HEADS  # LSTM_CELLS // NUM_HEADS

    # ---------------------------
    # Decoder inputs
    # ---------------------------
    dec_in = Input(shape=(LOG_MEL_MAX_LENGTH,), name='dec_in')
    last_h_1 = Input(shape=(M1_LSTM_CELLS,), name='last_h_1')
    last_c_1 = Input(shape=(M1_LSTM_CELLS,), name='last_c_1')
    last_h_2 = Input(shape=(M1_LSTM_CELLS,), name='last_h_2')
    last_c_2 = Input(shape=(M1_LSTM_CELLS,), name='last_c_2')

    # Encoder outputs
    enc_seq_out = Input(shape=(TEXT_MAX_LENGTH, M1_LSTM_CELLS), name='enc_seq_out')

    # ---------------------------
    # Embedding
    # ---------------------------
    emb_1 = Embedding(AUDIO_N_TOKENS, M1_DEC_EMBEDDING_DIM)(dec_in)

    emb_2 = parallel_cnn_1d_layers(
        input_tensor=emb_1,
        n_filters=[64, 32, 16, 8, 4],
        kernel_sizes=[3, 5, 7, 9, 15],
        concate_filters=M1_DEC_EMBEDDING_DIM * 1,
        skip_conn=False,
        name="dec_cnn_block_1",
        norm=False,
        activation='relu'
    )

    # ---------------------------
    # Decoder LSTMs
    # ---------------------------
    dec_out_1, dec_h_1, dec_c_1 = LSTM(
        M1_DEC_LSTM_CELLS,
        return_sequences=True,
        return_state=True,
        name='decoder_lstm_1'
    )(emb_2, initial_state=[last_h_1, last_c_1])

    dec_out_2, dec_h_2, dec_c_2 = LSTM(
        M1_DEC_LSTM_CELLS,
        return_sequences=True,
        return_state=True,
        name='decoder_lstm_2'
    )(dec_out_1, initial_state=[last_h_2, last_c_2])

    # ---------------------------
    # Multi-Head Attention (decoder attends to encoder)
    # ---------------------------
    # Causal masking ensures decoder positions cannot see future tokens in self-attention
    mha = MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=KEY_DIM,
        name='decoder_mha'
    )

    attention_context = mha(
        query=dec_out_2,
        value=enc_seq_out,
        key=enc_seq_out,
        use_causal_mask=True  # Ensures no future decoder tokens are attended
    )

    # ---------------------------
    # Concatenate context and decoder outputs
    # ---------------------------
    decoder_combined = Concatenate(axis=-1, name='decoder_combined')([dec_out_2, attention_context])

    # ---------------------------
    # Final Dense layer
    # ---------------------------
    decoder_output_final = Dense(AUDIO_N_TOKENS, activation='softmax', name='decoder_output_final')(decoder_combined)

    # ---------------------------
    # Decoder model
    # ---------------------------
    decoder_model = Model(
        [dec_in, last_h_1, last_c_1, last_h_2, last_c_2, enc_seq_out],
        [decoder_output_final, dec_h_1, dec_c_1, dec_h_2, dec_c_2],
        name='decoder_model'
    )

    decoder_model.summary()
    return decoder_model

In [ ]:
def build_m1():
    NAME = "trail_3"

    encoder_input = Input(shape=(TEXT_MAX_LENGTH,), name='entire_model_input')
    decoder_teacher_input = Input(shape=(LOG_MEL_MAX_LENGTH,), name='entire_teacher_input')

    encoder = build_m1_encoder()
    decoder = build_m1_decoder()

    enc_out_2, enc_h_1, enc_c_1, enc_h_2, enc_c_2 = encoder(encoder_input)

    decoder_outputs, _, _, _, _ = decoder([
        decoder_teacher_input,
        enc_h_1,
        enc_c_1,
        enc_h_2,
        enc_c_2,
        enc_out_2
    ])

    entire_model = Model(
        inputs=[encoder_input, decoder_teacher_input],
        outputs=decoder_outputs,
        name='entire_model'
    )

    entire_model.compile(
        optimizer='adam',
        loss=masked_categorical_crossentropy,
        metrics=[
            tf.keras.metrics.MeanMetricWrapper(masked_categorical_accuracy, name='masked_acc')
        ]
    )

    entire_model.summary()

    models_list.append(CustomModel(
        name=NAME,
        entire_model=entire_model,
        encoder_model=encoder,
        decoder_model=decoder
        ))

    return entire_model

In [ ]:
build_m1()

## Plotting

In [ ]:
for model in models_list:
      keras_model = model.entire_model

      img_path = os.path.join(TRAIN_CACHE_DIR,f'{model.name}_arch.png')

      plot_model(
          keras_model,
          to_file= img_path,
          show_shapes=True,
          dpi =200,
          show_layer_names=True,
          expand_nested=True,
          show_layer_activations=True,
          show_trainable=True,
          show_dtype=True,
      )

# Training

In [ ]:
for custom_model in models_list:
  print(f"Training {custom_model.name}")

  custom_model.fit(
      data_gen = train_data_gen,
      # data_gen = samples_data_gen,
      validation_data = None,
      epochs = 50,
      force_train = True
  )

# History

In [ ]:
plt.style.use(plt.style.available[12])

for custom_model in models_list:
  hist = custom_model.history

  loss = hist['loss']
  masked_acc = hist['masked_acc']

  plt.figure(figsize=(12, 4))
  plt.title(f"Model: {custom_model.name}")

  plt.subplot(1, 2, 1)
  plt.plot(loss, label='Training Loss')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.legend()

  plt.subplot(1, 2, 2)
  plt.plot(masked_acc, label='Training Acc (masked)')
  plt.xlabel('Epoch')
  plt.ylabel('ACC')
  plt.legend()

  plt.tight_layout()
  plt.show()

# Inference

In [ ]:
def inference_txt(
    txt: str,
    max_steps=None,
    custom_model=models_list[0],
):
    if not max_steps:
      max_steps = LOG_MEL_MAX_LENGTH

    # -----------------------
    # 1. Text preprocessing
    # -----------------------
    x_pad_id = tokens_dict[TOKEN_PAD]

    txt_cleaned = process_txt(txt)

    tokens = tokenize(txt_cleaned)

    padded_tokens = np.pad(tokens,
         (0, TEXT_MAX_LENGTH - len(tokens)),
         'constant',
         constant_values=x_pad_id)

    batch_tokens = tf.expand_dims(padded_tokens, axis=0)

    encoder = custom_model.encoder_model
    decoder = custom_model.decoder_model

    # -----------------------
    # 2. Encoder forward pass
    # -----------------------
    (encoder_outputs,
     h1, c1,
     h2, c2) = encoder(batch_tokens, training=False)

    # -----------------------
    # 3. Initial decoder input (start frame)
    # -----------------------
    current_input = tf.convert_to_tensor([[ audio_tokens_dict[TOKEN_START] ]], dtype=tf.int32)

    generated_tokens = []

    # -----------------------
    # 4. Autoregressive loop
    # -----------------------
    for step in tqdm.tqdm(range(max_steps)):

        (prediction,
         h1, c1,
         h2, c2) = decoder([
            current_input,
            h1, c1,
            h2, c2,
            encoder_outputs
        ], training=False)

        # prediction
        token_prob = prediction.numpy().flatten()
        audio_token_idx = np.argmax(token_prob)
        audio_token = reverse_audio_tokens_dict[audio_token_idx]

        # Append frame
        generated_tokens.append(audio_token)

        # Feed next step
        current_input = tf.convert_to_tensor([[audio_token_idx]], dtype=tf.int32)

    print('\n\n')
    print("Cleaned_txt:", txt_cleaned)
    print("Total prediction:", len(generated_tokens))
    print("Predicted(first 100):", ' '.join(generated_tokens[:100]))

    filtered_generated_tokens = [token for token in generated_tokens if token != TOKEN_PAD]
    filtered_generated_tokens = [token for token in generated_tokens if token != TOKEN_START]
    filtered_generated_tokens = [token for token in generated_tokens if token != TOKEN_END]

    log_mel = audio_tokens_2_log_mels_values(filtered_generated_tokens)
    audio_series = log_mel_2_audio(log_mel)

    librosa.display.specshow(log_mel, sr=AUDIO_SR, hop_length=AUDIO_HOP_LENGTH,
                         x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.2f dB')
    plt.title('Quantized Log-Mel Spectrogram')
    plt.tight_layout()
    plt.show()

    display(AudioPlayer(audio_series, rate=AUDIO_SR))


In [ ]:
inference_txt("اهلا", max_steps=25)

In [ ]:
inference_txt(data_df.loc[0, 'cleaned_txt'], max_steps=250)

In [ ]:
inference_txt(data_df.loc[1, 'cleaned_txt'], max_steps=250)